In [4]:
! python -m pip install selenium webdriver-manager
! python -m pip install requests pandas beautifulsoup4

  Using cached selenium-4.39.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached trio-0.32.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached selenium-4.39.0-py3-none-any.whl (9.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [8]:
# 1. 설정
TARGET_URL = "https://defense.na.go.kr:444/cmmit/bbs/B0000051/list.do?pageIndex=1&menuNo=2000037&searchWrd=&searchCnd=1&sdate=&edate=&searchWrdMb=&sdateMb=&edateMb=&pageUnit=50"
SAVE_DIR = os.path.abspath("defense_files")

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# 크롬 옵션 설정 (보안 강화 사이트 대응)
chrome_options = Options()
prefs = {
    "download.default_directory": SAVE_DIR,
    "download.prompt_for_download": False,
    "directory_upgrade": True,
    "safebrowsing.enabled": False # 안전하지 않은 파일 경고 무시
}
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument('--ignore-certificate-errors') # SSL 인증서 오류 무시
chrome_options.add_argument('--ignore-ssl-errors')

# 드라이버 실행
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

def download_files():
    try:
        driver.get(TARGET_URL)
        
        # 페이지 로딩을 확실히 기다리기 위해 명시적 대기 사용
        wait = WebDriverWait(driver, 10)
        # 테이블 몸체(tbody)가 나타날 때까지 대기
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "tbody")))
        
        # 행(tr)을 찾는 선택자를 더 넓게 잡음
        rows = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
        print(f"총 {len(rows)}개의 항목을 발견했습니다.")

        if len(rows) == 0:
            print("데이터를 찾지 못했습니다. 브라우저 창에 페이지가 정상적으로 뜨는지 확인해주세요.")
            return

        for i in range(len(rows)):
            try:
                # 매 반복마다 요소를 새로 갱신 (StaleElementReferenceException 방지)
                current_rows = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
                row = current_rows[i]
                
                # 다운로드 버튼 찾기
                download_btn = row.find_element(By.CSS_SELECTOR, "a.btn_board_download")
                file_info = download_btn.get_attribute("title")
                print(f"[{i+1}/{len(rows)}] {file_info} 처리 중...")

                # 1. 자바스크립트 직접 실행하여 레이어 열기
                driver.execute_script("arguments[0].click();", download_btn)
                time.sleep(1) # 레이어 로딩 대기

                # 2. 레이어 내부의 실제 다운로드 링크 클릭
                # 이미지에 기반하여 div.board_download_layer 내부의 a 태그 탐색
                real_links = row.find_elements(By.CSS_SELECTOR, ".board_download_layer a")
                
                for link in real_links:
                    driver.execute_script("arguments[0].click();", link)
                    time.sleep(1) # 파일별 다운로드 간격
                
            except Exception as e:
                # 게시물에 첨부파일이 없는 경우 등 예외 처리
                continue

        print(f"\n✅ 작업 완료! 파일 저장 경로: {SAVE_DIR}")
        time.sleep(5) # 마지막 다운로드 완료 대기

    finally:
        driver.quit()

if __name__ == "__main__":
    download_files()

총 38개의 항목을 발견했습니다.
[1/38] [사진자료] 국회 국방위원회 남수단 한빛부대 현장방문 사진(파일 다운로드 대화상자 열기) 처리 중...
[2/38] 국회 국방위원회, 남수단 한빛부대 현장방문(파일 다운로드 대화상자 열기) 처리 중...
[3/38] 소령 정년, 45세에서 50세로 5년 연장(파일 다운로드 대화상자 열기) 처리 중...
[4/38] 국방위원회, 한미연합군사령부 전시지휘소 현장방문(파일 다운로드 대화상자 열기) 처리 중...
[5/38] 국회 국방위, 軍 출산·육아 지원 정책 개선 간담회(파일 다운로드 대화상자 열기) 처리 중...
[6/38] 국회 국방위, 유럽의회 대표단 면담(파일 다운로드 대화상자 열기) 처리 중...
[7/38] 국회 국방위 대표단, K-방산 세일즈 외교 성공적으로 마치고 귀국(파일 다운로드 대화상자 열기) 처리 중...
[8/38] 국회 국방위, 한국형 3축 체계 구축을 위한 예산 의결(파일 다운로드 대화상자 열기) 처리 중...
[9/38] 국회 국방위, 북한의 탄도미사일 도발 규탄 및 중단 촉구(파일 다운로드 대화상자 열기) 처리 중...
[10/38] 국회 국방위, 전임 한미연합사령관 3인 면담(파일 다운로드 대화상자 열기) 처리 중...
[11/38] 국회 국방위, NATO 의회연맹 대표단 면담(파일 다운로드 대화상자 열기) 처리 중...
[12/38] 국회 국방위 병역법 개정관련 국민 여론조사 실시(파일 다운로드 대화상자 열기) 처리 중...
[13/38] 소관 부처의 2020회계연도 결산 의결(파일 다운로드 대화상자 열기) 처리 중...

✅ 작업 완료! 파일 저장 경로: c:\Users\user\OneDrive\Desktop\Desktop\VSCode\크롤링프로젝트\defense_files
